# LangGraph 15분 — 상태 하나를 노드가 차례로 고친다

3회차 수업 1~2부에서 씁니다. 셀 번호는 코드 셀 첫 줄의 `# [셀 N]` 표시입니다. 메뉴 Kernel → Restart Kernel and Run All Cells 를 하면 왼쪽의 `[N]` 도 같은 번호가 됩니다.

**이 노트북이 끝나면**

- 상태(State) · 노드(Node) · 엣지(Edge)가 무엇인지 코드로 가리킬 수 있습니다.
- 조건부 엣지로 "답변 / 거절"을 갈라 보낼 수 있습니다. 이게 **실습 1** 입니다.
- 오늘 채울 인제스천 그래프(`src/finrag/ingestion/graph.py`)가 이 노트북의 무엇과 같은지 압니다.

**필요 없는 것**: API 키, 임베딩 모델, 다운로드한 문서. 이 노트북은 LLM 을 한 번도 부르지 않고, 검색도 조항 3개짜리 장난감입니다.

> **왜 그래프인가.** 오늘 만드는 파이프라인은 직선이 아닙니다. 파싱 결과가 나쁘면 다른 파서로 돌아가고, 스캔본이면 OCR 캐시로 가고, 형식을 지원하지 않으면 사람 검토 큐로 갑니다. 이런 분기를 `if` 문으로 쌓으면 "어떤 문서가 어디로 갔는지"가 로그에만 남습니다. 그래프로 두면 지나온 경로가 그대로 기록이 됩니다.

In [1]:
# [셀 1] 준비
import sys
from importlib.metadata import version

print("python        ", sys.version.split()[0])
for pkg in ("langgraph", "langchain-core"):
    print(f"{pkg:14}", version(pkg))

python         3.12.14
langgraph      1.2.12
langchain-core 1.6.6


## 1. State — 노드들이 주고받는 하나의 딕셔너리

그래프는 **상태 하나**를 노드들이 차례로 고쳐 나가는 구조입니다. 상태는 딕셔너리이고, 어떤 키가 있는지를 `TypedDict` 로 적어 둡니다.

- `total=False` 를 붙이는 이유: 처음부터 모든 키가 있지는 않습니다. `answer` 는 마지막 노드가 채웁니다.
- 여기 적히지 않은 키는 노드가 반환해도 **버려집니다.** 오늘 코드의 `IngestState` 도 같은 규칙입니다.

In [2]:
# [셀 2] State
from typing import Literal, TypedDict


class State(TypedDict, total=False):
    question: str        # 사용자가 물은 것
    hits: list[dict]     # 검색된 조항들
    grade: str           # 근거가 충분한가: sufficient / none
    answer: str          # 최종 답변
    trace: list[str]     # 어느 노드를 지나왔는가. 경로 자체가 기록입니다


print("State 의 키:", list(State.__annotations__))

State 의 키: ['question', 'hits', 'grade', 'answer', 'trace']


## 2. 장난감 검색기

실제 검색은 오늘 4부(Qdrant)와 4회차에 만듭니다. 여기서는 조항 3개짜리 장난감을 씁니다. 글자 두 개씩 잘라서 겹치는 개수를 세는 방식이라, "코퍼스에 있는 질문"과 "없는 질문"이 갈리는 것만 보면 됩니다.

In [3]:
# [셀 3] 장난감 검색기
CORPUS = [
    {"doc_id": "toy_deposit", "article": "제10조(중도해지이율)",
     "text": "만기일 이전에 해지하는 경우 가입일 당시 영업점에 게시한 중도해지이율을 적용합니다."},
    {"doc_id": "toy_deposit", "article": "제3조(가입대상)",
     "text": "실명의 개인으로 하며 1인 1계좌에 한하여 가입할 수 있습니다."},
    {"doc_id": "toy_loan", "article": "제7조(중도상환수수료)",
     "text": "대출일부터 3년 이내에 상환하는 경우 상환금액에 수수료율을 곱한 금액을 부과합니다."},
]

MIN_OVERLAP = 2      # 겹치는 글자쌍이 이보다 적으면 검색 결과에서 뺍니다


def bigrams(s: str) -> set[str]:
    s = "".join(s.split())
    return {s[i:i + 2] for i in range(len(s) - 1)}


def search(query: str, k: int = 2) -> list[dict]:
    scored = [(len(bigrams(query) & bigrams(c["article"] + c["text"])), c) for c in CORPUS]
    scored.sort(key=lambda t: -t[0])
    return [c for n, c in scored if n >= MIN_OVERLAP][:k]


for q in ["중도해지이율은 어떻게 적용되나요?", "암 진단비는 얼마인가요?"]:
    print(q, "→", [c["article"] for c in search(q)] or "검색 결과 없음")

중도해지이율은 어떻게 적용되나요? → ['제10조(중도해지이율)']
암 진단비는 얼마인가요? → 검색 결과 없음


## 3. Node — 상태를 받아 바뀐 부분만 반환하는 함수

노드는 특별한 클래스가 아니라 **그냥 함수**입니다. 규칙은 두 개뿐입니다.

1. 인자로 상태를 받습니다.
2. 바뀐 키만 담은 딕셔너리를 반환합니다. LangGraph 가 원래 상태에 합쳐 줍니다.

`trace` 를 `state.get("trace", []) + [...]` 로 이어 붙이는 것을 봐 두세요. 오늘 코드의 `warnings` 도 같은 방식으로 쌓입니다.

In [4]:
# [셀 4] 노드 네 개
def retrieve(state: State) -> State:
    hits = search(state["question"])
    return {"hits": hits, "trace": state.get("trace", []) + ["retrieve"]}


def grade(state: State) -> State:
    """근거가 쓸 만한가. 오늘 코드에서는 이 자리에 검증 게이트가 들어갑니다."""
    verdict = "sufficient" if state.get("hits") else "none"
    return {"grade": verdict, "trace": state.get("trace", []) + ["grade"]}


def answer(state: State) -> State:
    h = state["hits"][0]
    return {"answer": f"{h['article']}에 따르면, {h['text']}  [근거] {h['doc_id']} · {h['article']}",
            "trace": state.get("trace", []) + ["answer"]}


def abstain(state: State) -> State:
    """근거가 없으면 답을 만들지 않습니다. 이 수업에서 가장 중요한 노드입니다."""
    return {"answer": "제공된 문서에서 근거를 찾지 못했습니다. 다른 표현으로 물어보시겠어요?",
            "trace": state.get("trace", []) + ["abstain"]}


print(grade({"hits": [CORPUS[0]]}))
print(grade({"hits": []}))

{'grade': 'sufficient', 'trace': ['grade']}
{'grade': 'none', 'trace': ['grade']}


## 4. 조건부 엣지 — 실습 1

`retrieve → grade` 는 항상 같은 방향이라 보통 엣지로 잇습니다. `grade` 다음은 상태에 따라 갈립니다. 이때 쓰는 것이 **라우터 함수**입니다.

라우터는 노드가 아닙니다. 상태를 **읽기만** 하고 다음 목적지의 **이름(문자열)** 을 반환합니다. 반환형에 `Literal` 을 적어 두면 목적지 이름에 오타가 나도 편집기가 잡아 줍니다.

**실습 1.** 아래 `route_after_grade` 를 채웁니다. `grade` 가 `"sufficient"` 면 `"answer"`, 아니면 `"abstain"` 을 반환합니다. 한 줄이면 됩니다. 셀을 실행해서 "실습 1 통과"가 나오면 끝입니다.

오늘 코드의 `nodes.route_after_gate` 가 정확히 이 모양입니다. 거기서는 `state["route"]` 를 그대로 반환합니다.

In [8]:
# [셀 5] 실습 1 - 라우터 함수
def route_after_grade(state: State) -> Literal["answer", "abstain"]:
    # grade 가 "sufficient" 면 "answer", 아니면 "abstain"
    return "answer" if state.get("grade") == "sufficient" else "abstain"


assert route_after_grade({"grade": "sufficient"}) == "answer", "근거가 충분하면 answer 로 보내야 합니다"
assert route_after_grade({"grade": "none"}) == "abstain", "근거가 없으면 abstain 으로 보내야 합니다"
assert route_after_grade({}) == "abstain", "grade 가 아예 없어도 abstain 입니다"

print("실습 1 통과")

실습 1 통과


## 5. 조립하고 그림 보기

노드를 등록하고, 시작점을 정하고, 엣지를 잇고, `compile()` 합니다. 조건부 엣지는 `add_conditional_edges(노드, 라우터, {라우터가 돌려주는 값: 다음 노드})` 입니다.

`compile()` 한 그래프는 mermaid 텍스트로 그림을 그려 줍니다. 그림을 손으로 그리지 않아서 코드와 그림이 어긋나지 않습니다.

In [10]:
# [셀 6] 조립
from langgraph.graph import END, StateGraph


def build():
    g = StateGraph(State)
    g.add_node("retrieve", retrieve)
    g.add_node("grade", grade)
    g.add_node("answer", answer)
    g.add_node("abstain", abstain)

    g.set_entry_point("retrieve")
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges("grade", route_after_grade,
                            {"answer": "answer", "abstain": "abstain"})
    g.add_edge("answer", END)
    g.add_edge("abstain", END)
    return g.compile()


graph = build()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade(grade)
	answer(answer)
	abstain(abstain)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade -.-> abstain;
	grade -.-> answer;
	retrieve --> grade;
	abstain --> __end__;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 6. 실행 — 답변과 거절이 갈리는 것을 본다

같은 그래프에 질문 두 개를 넣습니다. 하나는 코퍼스에 있고 하나는 없습니다. **경로(trace)가 다르게 찍히는 것**이 핵심입니다. 오늘 인제스천 리포트의 "경로별 문서 수"가 바로 이 trace 를 문서 51건에 대해 센 것입니다.

In [11]:
# [셀 7] 실행
for q in ["중도해지이율은 어떻게 적용되나요?", "암 진단비는 얼마인가요?"]:
    out = graph.invoke({"question": q})
    print("Q.", q)
    print("   경로:", " → ".join(out["trace"]))
    print("   답 :", out["answer"][:60])
    print()

Q. 중도해지이율은 어떻게 적용되나요?
   경로: retrieve → grade → answer
   답 : 제10조(중도해지이율)에 따르면, 만기일 이전에 해지하는 경우 가입일 당시 영업점에 게시한 중도해지이율을 적

Q. 암 진단비는 얼마인가요?
   경로: retrieve → grade → abstain
   답 : 제공된 문서에서 근거를 찾지 못했습니다. 다른 표현으로 물어보시겠어요?



## 7. stream — 노드마다 무엇이 바뀌었는지 본다

`invoke` 는 최종 상태만 줍니다. `stream` 은 노드가 끝날 때마다 **그 노드가 돌려준 부분 딕셔너리**를 흘려 줍니다. 노드가 "바뀐 키만" 반환한다는 규칙이 여기서 눈에 보입니다.

In [12]:
# [셀 8] stream
for step in graph.stream({"question": "중도상환수수료는 언제 내나요?"}):
    for node, patch in step.items():
        print(f"[{node:9}] 바뀐 키: {sorted(patch)}")

[retrieve ] 바뀐 키: ['hits', 'trace']
[grade    ] 바뀐 키: ['grade', 'trace']
[answer   ] 바뀐 키: ['answer', 'trace']


## 8. 오늘 코드에서는 어디에 있나

| 이 노트북 | 오늘 코드 (`src/finrag/ingestion/`) |
|---|---|
| `State` | `nodes.IngestState` — doc_id, pages, report, route, chunks … |
| 노드 함수 `retrieve`, `grade` … | `nodes.classify`, `parse`, `gate`(실습 2), `reparse`, `ocr`, `llm_repair`, `chunk`, `dead_letter` |
| 라우터 `route_after_grade`(실습 1) | `nodes.route_after_gate`, `route_after_reparse`, `route_after_ocr` |
| `build()` 의 엣지 연결 | `graph.build_ingest_graph`(실습 3) |
| `graph.invoke({"question": …})` | `pipelines/ingest.py` 가 문서마다 `invoke({"doc_id": …, "path": …})` |

아래 셀은 오늘 코드의 그래프 그림입니다. 실습 3 을 채우기 전에는 "아직 채우지 않았습니다"가 나오고, 채우고 나서 다시 실행하면 그림이 나옵니다.

In [13]:
# [셀 9] 인제스천 그래프 그림 (실습 3 뒤에 다시 실행)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else "src")

try:
    from finrag.ingestion.graph import build_ingest_graph
    print(build_ingest_graph().get_graph().draw_mermaid())
except NotImplementedError as e:
    print("아직 채우지 않았습니다:", e)

아직 채우지 않았습니다: 실습 3: build_ingest_graph 의 엣지를 연결하세요 (이 줄은 지운다)
